# DramaBox — Expressive TTS with Voice Cloning

This notebook demonstrates how to use DramaBox for text-to-speech generation.

**Requirements:**
- A CUDA-capable GPU with ~24GB VRAM
- DramaBox installed in the `.venv` of this directory
- (Optional) A 10+ second voice reference audio file

In [ ]:
# Download models and initialize the TTS server on CUDA
import os
# add C:\Users\rob\Desktop\tts-generation-webui-main\installer_files\env\Library\bin\ to PATH
os.environ["PATH"] += os.pathsep + r"C:\\Users\\rob\\Desktop\\tts-generation-webui-main\\installer_files\\env\\Library\\bin"

from dramabox.model_downloader import get_all_paths
from inference_server import TTSServer

PATHS = get_all_paths()
print("Models downloaded, initializing server...")
server = TTSServer(
    checkpoint=PATHS["transformer"],
    full_checkpoint=PATHS["audio_components"],
    gemma_root=PATHS["gemma_root"],
    device="cuda",
    dtype="bf16",
    compile_model=False,
    bnb_4bit=True,
)
print("TTSServer ready.")

In [1]:
# Download models and initialize the TTS server on CUDA
import sys, os
sys.path.insert(0, os.path.join((os.path.abspath(".")), "src"))
# print path
print(sys.path)
print(os.path.join((os.path.abspath(".")), "src"))
print(os.path.abspath("."))

# add C:\Users\rob\Desktop\tts-generation-webui-main\installer_files\env\Library\bin\ to PATH
os.environ["PATH"] += os.pathsep + r"C:\\Users\\rob\\Desktop\\tts-generation-webui-main\\installer_files\\env\\Library\\bin"

from model_downloader import get_all_paths
from inference_server import TTSServer

PATHS = get_all_paths()
print("Models downloaded, initializing server...")
server = TTSServer(
    checkpoint=PATHS["transformer"],
    full_checkpoint=PATHS["audio_components"],
    gemma_root=PATHS["gemma_root"],
    device="cuda",
    dtype="bf16",
    compile_model=False,
    bnb_4bit=True,
)
print("TTSServer ready.")

['c:\\Users\\rob\\Desktop\\temp\\tts-project-capture\\models\\DramaBox\\src', 'C:\\Users\\rob\\AppData\\Roaming\\uv\\python\\cpython-3.10.11-windows-x86_64-none\\python310.zip', 'C:\\Users\\rob\\AppData\\Roaming\\uv\\python\\cpython-3.10.11-windows-x86_64-none\\DLLs', 'C:\\Users\\rob\\AppData\\Roaming\\uv\\python\\cpython-3.10.11-windows-x86_64-none\\lib', 'C:\\Users\\rob\\AppData\\Roaming\\uv\\python\\cpython-3.10.11-windows-x86_64-none', 'c:\\Users\\rob\\Desktop\\temp\\tts-project-capture\\models\\DramaBox\\.venv', '', 'c:\\Users\\rob\\Desktop\\temp\\tts-project-capture\\models\\DramaBox\\.venv\\lib\\site-packages']
c:\Users\rob\Desktop\temp\tts-project-capture\models\DramaBox\src
c:\Users\rob\Desktop\temp\tts-project-capture\models\DramaBox


c:\Users\rob\Desktop\temp\tts-project-capture\models\DramaBox\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-19 12:29:04,355 INFO Fetching transformer from ResembleAI/Dramabox/dramabox-dit-v1.safetensors...
2026-05-19 12:29:04,581 INFO   -> C:\Users\rob\.cache\dramabox\models--ResembleAI--Dramabox\snapshots\404f967f653fa1170dc15a9d1ddd3fdb9a0a842d\dramabox-dit-v1.safetensors
2026-05-19 12:29:04,581 INFO Fetching audio_components from ResembleAI/Dramabox/dramabox-audio-components.safetensors...
2026-05-19 12:29:04,730 INFO   -> C:\Users\rob\.cache\dramabox\models--ResembleAI--Dramabox\snapshots\404f967f653fa1170dc15a9d1ddd3fdb9a0a842d\dramabox-audio-components.safetensors
2026-05-19 12:29:04,731 INFO Fetching silence_latent from ResembleAI/Dramabox/assets/silence_latent_frame.pt...
2026-05-19 12:29:0

Models downloaded, initializing server...


W0519 12:29:05.798000 32044 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.44s/it]
2026-05-19 12:29:19,579 INFO Gemma 4-bit loaded: 7.8GB VRAM
2026-05-19 12:29:21,630 WARNING Uninitialized parameters or buffers: ['feature_extractor.video_aggregate_embed.weight', 'feature_extractor.video_aggregate_embed.bias', 'video_connector.learnable_registers', 'video_connector.transformer_1d_blocks.0.attn1.q_norm.weight', 'video_connector.transformer_1d_blocks.0.attn1.k_norm.weight', 'video_connector.transformer_1d_blocks.0.attn1.to_q.weight', 'video_connector.transformer_1d_blocks.0.attn1.to_q.bias', 'video_connector.transformer_1d_blocks.0.attn1.to_k.weight', 'video_connector.transformer_1d_blocks.0.attn1.to_k.bias', 'video_connector.transformer_1d_blocks.0.attn1.to_v.weight', 'video_connector.transformer_1d_blocks.0.attn1.to_v.bias', 'video_connector.transformer_1d

TTSServer ready.


In [5]:
import torch
import torchcodec

print(f"Torch version: {torch.__version__}")
print(f"TorchCodec imported successfully!")

# Basic Smoke Test: Decoding a video
from torchcodec.decoders import VideoDecoder

# Create a sample black video (1 second, 25 fps, 360x640) for testing
# (Or replace with the path to any local .mp4 video file)
height, width = 360, 640
fps = 25
num_frames = 25
raw_video_bytes = torch.zeros(num_frames, 3, height, width, dtype=torch.uint8) 

# Initialize the VideoDecoder
try:
    decoder = VideoDecoder(raw_video_bytes, pts_per_second=fps, height=height, width=width)
    print(f"Total video duration: {decoder.metadata.duration_seconds} seconds")

    # Decode the first frame
    frame = decoder[0]
    print(f"Frame 1 decoded successfully!")
    print(f"Decoded Tensor Shape: {frame.data.shape}")
    print("Smoke test passed!")
except Exception as e:
    print(f"Smoke test failed: {e}")


Torch version: 2.11.0+cu128
TorchCodec imported successfully!
Smoke test failed: VideoDecoder.__init__() got an unexpected keyword argument 'pts_per_second'


In [ ]:
from IPython.display import Audio

waveform, sr = server.generate(
    prompt='A woman speaks warmly, "Hello, how are you today?" She laughs, "Hahaha, it is so good to see you!"',
    voice_ref=None,
    cfg_scale=2.5,
    stg_scale=1.5,
    duration_multiplier=1.1,
    seed=41,
    ref_duration=10.0,
    rescale_scale="auto",
    gen_duration=30,
)

Audio(waveform.cpu().numpy(), rate=sr)


2026-05-19 12:45:18,507 INFO Prompt: 1.32s
2026-05-19 12:45:18,508 INFO Auto rescale_scale = 0.30 for cfg=2.5
100%|██████████| 30/30 [00:11<00:00,  2.59it/s]
2026-05-19 12:45:30,109 INFO Denoise (30 steps): 11.59s
2026-05-19 12:45:30,543 INFO Decode: 0.43s
2026-05-19 12:45:30,543 INFO Total: 13.36s for 30.1s audio


## Prompt Writing Tips

**Structure:** `<speaker description>, "<dialogue>" <action direction> "<more dialogue>"`

**Inside quotes** (model produces actual sounds):
- Laughs: `"Hahaha"` `"Hehehe"` (always one word, never separated)
- Sounds: `"Mmmmm"` `"Ugh"` `"Argh"` `"Ahhh"` `"Hmm"`

**Outside quotes** (stage directions):
- `She sighs deeply.` · `He gulps nervously.` · `A long pause.`
- `Her voice cracks.` · `He clears his throat.` · `She scoffs.`

**Avoid inside quotes** (model speaks them literally): `Ahem`, `Pfft`, `Sigh`, `Gasp`, `Cough`.

## Inference Settings

| Parameter | Default | Notes |
|---|---|---|
| `cfg-scale` | 2.5 | Lower = more natural, higher = more text-faithful |
| `stg-scale` | 1.5 | Skip-token guidance |
| `rescale` | 0 | No rescaling |
| `modality` | 1 | No modality guidance |
| `duration-multiplier` | 1.1 | 10% breathing room on auto-estimated length |
| `steps` | 30 | Euler flow matching |

In [ ]:
# Verify watermark detection (every output is automatically watermarked)
import perth, librosa

wav, sr = librosa.load("output.wav", sr=None, mono=True)
detector = perth.PerthImplicitWatermarker()
print(detector.get_watermark(wav, sample_rate=sr))   # confidence ≈ 1.0